In [4]:
from pyspark.sql import SparkSession
import random

spark = SparkSession.builder \
    .appName("Week5_Assignment") \
    .getOrCreate()

data = []

cities = ["Delhi","Mumbai","Bangalore","Hyderabad","Chennai","Pune"]
regions = ["North","South","East","West"]
categories = ["Electronics","Clothing","Furniture","Groceries"]
subscriptions = ["Premium","Basic"]
status_list = ["Active","Inactive",None]

for i in range(1,1001):
    amount = float(f"{random.uniform(100,5000):.2f}")

    data.append((
        i,
        random.randint(18,60),
        random.choice(cities),
        random.choice(regions),
        random.choice(categories),
        amount,
        random.choice(subscriptions),
        random.choice(status_list),
        f"user{i}@mail.com" if random.random() > 0.05 else None,
        f"user{i}" if random.random() > 0.05 else "",
        f"2025-06-{random.randint(1,28)}",
        random.randint(1,20)
    ))

columns = [
    "user_id","age","city","region",
    "product_category","sale_amount",
    "subscription","status",
    "email","username",
    "transaction_date","store_id"
]

df = spark.createDataFrame(data, columns)

df.show(5)

+-------+---+---------+------+----------------+-----------+------------+--------+--------------+--------+----------------+--------+
|user_id|age|     city|region|product_category|sale_amount|subscription|  status|         email|username|transaction_date|store_id|
+-------+---+---------+------+----------------+-----------+------------+--------+--------------+--------+----------------+--------+
|      1| 52|Bangalore| South|       Furniture|     1576.4|     Premium|Inactive|user1@mail.com|   user1|      2025-06-14|      18|
|      2| 37|     Pune| South|        Clothing|     4683.6|       Basic|  Active|user2@mail.com|   user2|      2025-06-12|      10|
|      3| 48|  Chennai|  West|       Groceries|     3459.8|       Basic|  Active|user3@mail.com|   user3|       2025-06-5|      13|
|      4| 59|Hyderabad|  West|       Groceries|    4479.36|     Premium|    NULL|user4@mail.com|   user4|       2025-06-4|       4|
|      5| 33|Hyderabad|  East|     Electronics|    4915.98|     Premium|    

## Q1. Limitations of Traditional MapReduce

Traditional MapReduce has several limitations that make it less suitable for modern big data workloads:

- It relies heavily on disk-based processing, resulting in high I/O overhead.
- Intermediate results are written to disk after every stage, increasing execution time.
- Iterative algorithms require repeated disk access, making machine learning workloads slow.
- Complex programming model with more code development and maintenance effort.
- Limited support for real-time analytics and interactive queries.

### Why Spark is Preferred

Apache Spark overcomes these limitations through:

- In-memory computing
- Faster execution speed
- Simplified APIs
- Support for machine learning, streaming, and graph processing
- Better scalability for large datasets

## Q2. In-Memory Computing in Spark

Apache Spark uses in-memory computing to improve processing performance. Instead of repeatedly reading and writing intermediate data to disk, Spark stores frequently used datasets in memory.

This approach significantly reduces disk I/O operations and improves execution speed, especially for iterative machine learning algorithms where the same dataset is processed multiple times.

### Benefits

- Faster data access
- Reduced disk dependency
- Improved performance for iterative computations
- Better support for machine learning and analytics workloads

In [5]:
# Remove duplicates using user_id and transaction_date

df_clean = df.dropDuplicates(
    ["user_id","transaction_date"]
)

df_clean.show()

+-------+---+---------+------+----------------+-----------+------------+--------+---------------+--------+----------------+--------+
|user_id|age|     city|region|product_category|sale_amount|subscription|  status|          email|username|transaction_date|store_id|
+-------+---+---------+------+----------------+-----------+------------+--------+---------------+--------+----------------+--------+
|      1| 52|Bangalore| South|       Furniture|     1576.4|     Premium|Inactive| user1@mail.com|   user1|      2025-06-14|      18|
|      2| 37|     Pune| South|        Clothing|     4683.6|       Basic|  Active| user2@mail.com|   user2|      2025-06-12|      10|
|      3| 48|  Chennai|  West|       Groceries|     3459.8|       Basic|  Active| user3@mail.com|   user3|       2025-06-5|      13|
|      4| 59|Hyderabad|  West|       Groceries|    4479.36|     Premium|    NULL| user4@mail.com|   user4|       2025-06-4|       4|
|      5| 33|Hyderabad|  East|     Electronics|    4915.98|     Premi

In [6]:
# Calculate average sales category-wise for West region

df_sales = df.filter(
    col("region") == "West"
)

df_sales.groupBy(
    "product_category"
).agg(
    avg("sale_amount").alias("avg_sales")
).show()

+----------------+------------------+
|product_category|         avg_sales|
+----------------+------------------+
|       Groceries| 2572.757547169811|
|     Electronics| 2419.558888888889|
|        Clothing|  2257.94696969697|
|       Furniture|2928.7147999999997|
+----------------+------------------+



## Q5. Difference Between .na.drop() and .na.fill()

- **.na.drop()** removes rows containing null values.
- **.na.fill()** replaces null values with a specified value.



In [7]:
# Replace null status values

df = df.na.fill({
    "status":"Unknown"
})

df.show()

+-------+---+---------+------+----------------+-----------+------------+--------+---------------+--------+----------------+--------+
|user_id|age|     city|region|product_category|sale_amount|subscription|  status|          email|username|transaction_date|store_id|
+-------+---+---------+------+----------------+-----------+------------+--------+---------------+--------+----------------+--------+
|      1| 52|Bangalore| South|       Furniture|     1576.4|     Premium|Inactive| user1@mail.com|   user1|      2025-06-14|      18|
|      2| 37|     Pune| South|        Clothing|     4683.6|       Basic|  Active| user2@mail.com|   user2|      2025-06-12|      10|
|      3| 48|  Chennai|  West|       Groceries|     3459.8|       Basic|  Active| user3@mail.com|   user3|       2025-06-5|      13|
|      4| 59|Hyderabad|  West|       Groceries|    4479.36|     Premium| Unknown| user4@mail.com|   user4|       2025-06-4|       4|
|      5| 33|Hyderabad|  East|     Electronics|    4915.98|     Premi

In [8]:
# Count records by city

df.groupBy("city") \
  .count() \
  .filter(col("count") > 100) \
  .show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|  165|
|  Chennai|  184|
|   Mumbai|  149|
|     Pune|  178|
|    Delhi|  151|
|Hyderabad|  173|
+---------+-----+



## Q7. Immutability of Spark DataFrames

Spark DataFrames are immutable, meaning they cannot be modified after creation.

When performing data cleaning operations such as:

- Dropping columns
- Renaming columns
- Filtering rows
- Handling null values

Spark creates a new DataFrame instead of modifying the existing one.

### Advantages

- Prevents accidental data modification
- Supports fault tolerance
- Enables optimized execution planning

This design improves reliability and scalability in distributed environments.

In [9]:
new_df = df.drop("status")

In [10]:
# Filter premium users aged 18 to 30

df.filter(
    (col("age").between(18,30)) &
    (col("subscription") == "Premium")
).show()

+-------+---+---------+------+----------------+-----------+------------+--------+----------------+--------+----------------+--------+
|user_id|age|     city|region|product_category|sale_amount|subscription|  status|           email|username|transaction_date|store_id|
+-------+---+---------+------+----------------+-----------+------------+--------+----------------+--------+----------------+--------+
|     29| 19|   Mumbai| North|       Groceries|     218.05|     Premium|  Active| user29@mail.com|  user29|      2025-06-18|       7|
|     31| 18|Bangalore| North|     Electronics|    2534.55|     Premium| Unknown| user31@mail.com|  user31|       2025-06-7|       8|
|     37| 20|     Pune|  East|       Groceries|    3698.87|     Premium| Unknown| user37@mail.com|  user37|      2025-06-26|       7|
|     65| 19|Bangalore|  East|        Clothing|    1650.52|     Premium| Unknown| user65@mail.com|  user65|      2025-06-26|       9|
|     77| 22|   Mumbai| North|        Clothing|    2803.69|   

## Q9. Importance of Handling Null Values Before Aggregation

Null values can impact the accuracy of analytical results and may lead to incorrect calculations during aggregation operations.

Before applying functions such as:

- SUM()
- AVG()
- MIN()
- MAX()

it is important to clean or replace missing values.

### Benefits

- Improves calculation accuracy
- Prevents unexpected results
- Ensures data consistency
- Enhances reporting reliability

Proper null handling is a critical step in any data preprocessing workflow.

In [11]:
from pyspark.sql.types import TimestampType

# Convert raw timestamp to event time

df = df.withColumn(
    "event_time",
    col("transaction_date").cast(TimestampType())
)

df = df.drop("transaction_date")

df.show()

+-------+---+---------+------+----------------+-----------+------------+--------+---------------+--------+--------+-------------------+
|user_id|age|     city|region|product_category|sale_amount|subscription|  status|          email|username|store_id|         event_time|
+-------+---+---------+------+----------------+-----------+------------+--------+---------------+--------+--------+-------------------+
|      1| 52|Bangalore| South|       Furniture|     1576.4|     Premium|Inactive| user1@mail.com|   user1|      18|2025-06-14 00:00:00|
|      2| 37|     Pune| South|        Clothing|     4683.6|       Basic|  Active| user2@mail.com|   user2|      10|2025-06-12 00:00:00|
|      3| 48|  Chennai|  West|       Groceries|     3459.8|       Basic|  Active| user3@mail.com|   user3|      13|2025-06-05 00:00:00|
|      4| 59|Hyderabad|  West|       Groceries|    4479.36|     Premium| Unknown| user4@mail.com|   user4|       4|2025-06-04 00:00:00|
|      5| 33|Hyderabad|  East|     Electronics| 

## Q11. Shuffle Process in Spark

Shuffle is the process of redistributing data across partitions during operations such as:

- GroupBy
- Join
- Aggregation
- Distinct

During a shuffle, Spark moves data between executors to ensure that related records are grouped together.

### Why It Is a Wide Transformation

A transformation is considered wide when data needs to move across multiple partitions.

Since GroupBy operations require records with the same key to be collected in the same partition, Spark performs a shuffle operation.

### Impact

- Increased network communication
- Higher execution cost
- Additional memory usage

Therefore, shuffle operations are among the most expensive operations in Spark.

In [12]:
# Remove rows with null email or empty username

df_clean = df.filter(
    col("email").isNotNull()
).filter(
    col("username") != ""
)

df_clean.show()

+-------+---+---------+------+----------------+-----------+------------+--------+---------------+--------+--------+-------------------+
|user_id|age|     city|region|product_category|sale_amount|subscription|  status|          email|username|store_id|         event_time|
+-------+---+---------+------+----------------+-----------+------------+--------+---------------+--------+--------+-------------------+
|      1| 52|Bangalore| South|       Furniture|     1576.4|     Premium|Inactive| user1@mail.com|   user1|      18|2025-06-14 00:00:00|
|      2| 37|     Pune| South|        Clothing|     4683.6|       Basic|  Active| user2@mail.com|   user2|      10|2025-06-12 00:00:00|
|      3| 48|  Chennai|  West|       Groceries|     3459.8|       Basic|  Active| user3@mail.com|   user3|      13|2025-06-05 00:00:00|
|      4| 59|Hyderabad|  West|       Groceries|    4479.36|     Premium| Unknown| user4@mail.com|   user4|       4|2025-06-04 00:00:00|
|      5| 33|Hyderabad|  East|     Electronics| 

In [13]:
# Calculate min, max and average price

df.agg(
    min("sale_amount").alias("Min_Price"),
    max("sale_amount").alias("Max_Price"),
    avg("sale_amount").alias("Average_Price")
).show()

+---------+---------+------------------+
|Min_Price|Max_Price|     Average_Price|
+---------+---------+------------------+
|   103.14|  4998.07|2656.5715800000003|
+---------+---------+------------------+



## Q14. Risks of Using inferSchema=True

The inferSchema option automatically detects data types based on source data.

When datasets contain inconsistent or messy date formats, Spark may incorrectly infer the schema.

### Potential Risks

- Incorrect data type detection
- Date parsing errors
- Null values generated during conversion
- Inaccurate analytical results
- Data quality issues in downstream processes


In [14]:
# Final ETL Pipeline

final_report = (
    df.dropDuplicates()
      .na.fill({"sale_amount":0})
      .groupBy("store_id")
      .agg(
          sum("sale_amount")
          .alias("Total_Revenue")
      )
)

final_report.show()

+--------+------------------+
|store_id|     Total_Revenue|
+--------+------------------+
|      19|142136.74999999997|
|       7|132450.49999999997|
|       6|         142181.86|
|       9|106883.56000000001|
|      17|157920.30000000002|
|       5|135770.05000000005|
|       1|142918.66000000003|
|      10|         149928.19|
|       3|141073.48000000007|
|      12|         146285.63|
|       8| 99207.09000000001|
|      11| 96634.44000000003|
|       2|144076.26999999996|
|       4| 96214.87999999999|
|      13|143552.82000000007|
|      18|         131317.44|
|      14|110374.64000000003|
|      15|         172743.05|
|      20|122264.94999999998|
|      16|         142637.02|
+--------+------------------+



In [15]:
# Display final revenue report

final_report.orderBy(
    col("Total_Revenue").desc()
).show()

+--------+------------------+
|store_id|     Total_Revenue|
+--------+------------------+
|      15|         172743.05|
|      17|157920.30000000002|
|      10|         149928.19|
|      12|         146285.63|
|       2|144076.26999999996|
|      13|143552.82000000007|
|       1|142918.66000000003|
|      16|         142637.02|
|       6|         142181.86|
|      19|142136.74999999997|
|       3|141073.48000000007|
|       5|135770.05000000005|
|       7|132450.49999999997|
|      18|         131317.44|
|      20|122264.94999999998|
|      14|110374.64000000003|
|       9|106883.56000000001|
|       8| 99207.09000000001|
|      11| 96634.44000000003|
|       4| 96214.87999999999|
+--------+------------------+



In [16]:
# Verify successful pipeline execution

print("====================================")
print("ETL Pipeline Executed Successfully")
print("Data Cleaning Completed")
print("Transformations Applied")
print("Aggregations Generated")
print("Business Insights Extracted")
print("====================================")

ETL Pipeline Executed Successfully
Data Cleaning Completed
Transformations Applied
Aggregations Generated
Business Insights Extracted


In [17]:
# Project Summary Metrics

print(f"Total Records Processed : {df.count()}")
print(f"Unique Stores Analyzed : {df.select('store_id').distinct().count()}")
print("Pipeline Status : SUCCESS")

Total Records Processed : 1000
Unique Stores Analyzed : 20
Pipeline Status : SUCCESS


# Assignment Conclusion

This assignment successfully demonstrates an end-to-end Apache Spark data processing workflow. The implementation includes data cleaning, null handling, duplicate removal, filtering, aggregation, and reporting techniques commonly used in modern data engineering environments.

The final ETL pipeline transformed raw customer transaction data into actionable business insights while leveraging Spark's distributed processing capabilities. This project highlights practical experience in scalable data analytics, data quality management, and Spark-based ETL development.